<a href="https://colab.research.google.com/github/aminebadi3/PROJET-AAA/blob/main/PROJETAAA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Imports** :

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error ,r2_score
from sklearn.preprocessing import StandardScaler





#**Fusion de 'communities.data' et 'communities.names' :**

In [ ]:
#on extrait les noms des attributs de 'communities.names' pour nommer les colones de 'communities.data'
cols = [l.split()[1] for l in open("communities.names") if l.startswith("@attribute")]
# on remplace les '?' par NaN pour preciser qu'il s'agit des valeurs manquantes
data = pd.read_csv("communities.data", header=None, names=cols, na_values="?")



#**Target & Labels :**

In [ ]:
X=data.drop('ViolentCrimesPerPop',axis=1) #Labels
Y=data['ViolentCrimesPerPop'] #Target



#**Affichage des informations de la data avant nettoyage :**

In [ ]:
data.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Data columns (total 128 columns):
 #    Column                 Non-Null Count  Dtype  
---   ------                 --------------  -----  
 0    state                  1994 non-null   int64  
 1    county                 820 non-null    float64
 2    community              817 non-null    float64
 3    communityname          1994 non-null   object 
 4    fold                   1994 non-null   int64  
 5    population             1994 non-null   float64
 6    householdsize          1994 non-null   float64
 7    racepctblack           1994 non-null   float64
 8    racePctWhite           1994 non-null   float64
 9    racePctAsian           1994 non-null   float64
 10   racePctHisp            1994 non-null   float64
 11   agePct12t21            1994 non-null   float64
 12   agePct12t29            1994 non-null   float64
 13   agePct16t24            1994 non-null   float64
 14   agePct65up             1994 non-null  

In [ ]:
print(data.duplicated().sum())

0


In [ ]:
data.head()

,state,county,community,communityname,fold,population,householdsize,racepctblack,racePctWhite,racePctAsian,...,LandArea,PopDens,PctUsePubTrans,PolicCars,PolicOperBudg,LemasPctPolicOnPatr,LemasGangUnitDeploy,LemasPctOfficDrugUn,PolicBudgPerPop,ViolentCrimesPerPop
0,8,NaN,NaN,Lakewoodcity,1,0.19,0.33,0.02,0.90,0.12,...,0.12,0.26,0.20,0.06,0.04,0.9,0.5,0.32,0.14,0.20
1,53,NaN,NaN,Tukwilacity,1,0.00,0.16,0.12,0.74,0.45,...,0.02,0.12,0.45,NaN,NaN,NaN,NaN,0.00,NaN,0.67
2,24,NaN,NaN,Aberdeentown,1,0.00,0.42,0.49,0.56,0.17,...,0.01,0.21,0.02,NaN,NaN,NaN,NaN,0.00,NaN,0.43
3,34,5.0,81440.0,Willingborotownship,1,0.04,0.77,1.00,0.08,0.12,...,0.02,0.39,0.28,NaN,NaN,NaN,NaN,0.00,NaN,0.12
4,42,95.0,6096.0,Bethlehemtownship,1,0.01,0.55,0.02,0.95,0.09,...,0.04,0.09,0.02,NaN,NaN,NaN,NaN,0.00,NaN,0.03


#**Nettoyage de la data :**

On a decidé dans notre cas d'éliminer dans un premier temps toutes les colonnes qui ont moins de 1000 éléments

In [ ]:
X_drop = [col for col in X.columns if X[col].count() < 1000]

Les valeurs supprimés :

In [ ]:
X_drop

['county',
 'community',
 'LemasSwornFT',
 'LemasSwFTPerPop',
 'LemasSwFTFieldOps',
 'LemasSwFTFieldPerPop',
 'LemasTotalReq',
 'LemasTotReqPerPop',
 'PolicReqPerOffic',
 'PolicPerPop',
 'RacialMatchCommPol',
 'PctPolicWhite',
 'PctPolicBlack',
 'PctPolicHisp',
 'PctPolicAsian',
 'PctPolicMinor',
 'OfficAssgnDrugUnits',
 'NumKindsDrugsSeiz',
 'PolicAveOTWorked',
 'PolicCars',
 'PolicOperBudg',
 'LemasPctPolicOnPatr',
 'LemasGangUnitDeploy',
 'PolicBudgPerPop']

In [ ]:
X=X.drop(X_drop,axis=1)
X.info(verbose=True,show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Data columns (total 103 columns):
 #    Column                 Non-Null Count  Dtype  
---   ------                 --------------  -----  
 0    state                  1994 non-null   int64  
 1    communityname          1994 non-null   object 
 2    fold                   1994 non-null   int64  
 3    population             1994 non-null   float64
 4    householdsize          1994 non-null   float64
 5    racepctblack           1994 non-null   float64
 6    racePctWhite           1994 non-null   float64
 7    racePctAsian           1994 non-null   float64
 8    racePctHisp            1994 non-null   float64
 9    agePct12t21            1994 non-null   float64
 10   agePct12t29            1994 non-null   float64
 11   agePct16t24            1994 non-null   float64
 12   agePct65up             1994 non-null   float64
 13   numbUrban              1994 non-null   float64
 14   pctUrban               1994 non-null  

On remarque qu'il manque un élément à 'OtherPerCap' , on ajoute l'élément qui manque comme la medianne

In [ ]:
X['OtherPerCap'] = X['OtherPerCap'].fillna(X['OtherPerCap'].median())

In [ ]:
X.info(verbose=True,show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Data columns (total 103 columns):
 #    Column                 Non-Null Count  Dtype  
---   ------                 --------------  -----  
 0    state                  1994 non-null   int64  
 1    communityname          1994 non-null   object 
 2    fold                   1994 non-null   int64  
 3    population             1994 non-null   float64
 4    householdsize          1994 non-null   float64
 5    racepctblack           1994 non-null   float64
 6    racePctWhite           1994 non-null   float64
 7    racePctAsian           1994 non-null   float64
 8    racePctHisp            1994 non-null   float64
 9    agePct12t21            1994 non-null   float64
 10   agePct12t29            1994 non-null   float64
 11   agePct16t24            1994 non-null   float64
 12   agePct65up             1994 non-null   float64
 13   numbUrban              1994 non-null   float64
 14   pctUrban               1994 non-null  

Puisque notre projet se basera sur les caractéristiques socio-économiques , on éliminera les colonnes ['state' , 'communityname' , 'fold' , 'LemasPctOfficDrugUn'] .

Remarque : 'LemasPctOfficDrugUn' a été éliminé car elle appartient à la famille des colonnes à 319 éléments , et donc ne sera pas importante pour nous dans ce cas  

In [ ]:
X_drop2=["state" , "communityname" , "fold" , "LemasPctOfficDrugUn"]
X=X.drop(X_drop2,axis=1)


In [ ]:
X

,population,householdsize,racepctblack,racePctWhite,racePctAsian,racePctHisp,agePct12t21,agePct12t29,agePct16t24,agePct65up,...,NumInShelters,NumStreet,PctForeignBorn,PctBornSameState,PctSameHouse85,PctSameCity85,PctSameState85,LandArea,PopDens,PctUsePubTrans
0,0.19,0.33,0.02,0.90,0.12,0.17,0.34,0.47,0.29,0.32,...,0.04,0.00,0.12,0.42,0.50,0.51,0.64,0.12,0.26,0.20
1,0.00,0.16,0.12,0.74,0.45,0.07,0.26,0.59,0.35,0.27,...,0.00,0.00,0.21,0.50,0.34,0.60,0.52,0.02,0.12,0.45
2,0.00,0.42,0.49,0.56,0.17,0.04,0.39,0.47,0.28,0.32,...,0.00,0.00,0.14,0.49,0.54,0.67,0.56,0.01,0.21,0.02
3,0.04,0.77,1.00,0.08,0.12,0.10,0.51,0.50,0.34,0.21,...,0.00,0.00,0.19,0.30,0.73,0.64,0.65,0.02,0.39,0.28
4,0.01,0.55,0.02,0.95,0.09,0.05,0.38,0.38,0.23,0.36,...,0.00,0.00,0.11,0.72,0.64,0.61,0.53,0.04,0.09,0.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1989,0.01,0.40,0.10,0.87,0.12,0.16,0.43,0.51,0.35,0.30,...,0.00,0.00,0.22,0.28,0.34,0.48,0.39,0.01,0.28,0.05
1990,0.05,0.96,0.46,0.28,0.83,0.32,0.69,0.86,0.73,0.14,...,0.01,0.00,0.53,0.25,0.17,0.10,0.00,0.02,0.37,0.20
1991,0.16,0.37,0.25,0.69,0.04,0.25,0.35,0.50,0.31,0.54,...,0.06,0.02,0.25,0.68,0.61,0.79,0.76,0.08,0.32,0.18
1992,0.08,0.51,0.06,0.87,0.22,0.10,0.58,0.74,0.63,0.41,...,0.04,0.01,0.45,0.64,0.54,0.59,0.52,0.03,0.38,0.33
